##### <img style="float: left; padding-right: 10px; width: 250px" src="https://extension.harvard.edu/wp-content/uploads/sites/8/2020/08/image-10.png"> 



## CSCI S89 Deep Learning
## Name: Sergio Vazquez Lemus
## Final Project “Movie Recommendation System”

**Harvard University**<br/>
**Summer 2025**<br/>

<hr style="height:2pt">

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from IPython.display import Image
import os, shutil

In [ ]:
import tensorflow as tf
from keras import models
from keras import layers
from keras import optimizers
from keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D
from keras.preprocessing import image
from keras.models import load_model
from keras.utils import plot_model
from keras import models
from keras.layers import Input
from keras.models import Model
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense
from keras.callbacks import EarlyStopping
from keras import regularizers

In [ ]:
# Set seed for reproducibility
import random
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

## Overview 

The objective of this project is to develop a recommendation system that employs deep learning techniques to generate movie suggestions based on data consisting of users and their corresponding movie ratings. The aim is to construct a model capable of recommending movies by analyzing a user’s historical rating behavior.

For this project I used a dataset (ratings.dat) containing 1,000,209 anonymous ratings of more than 3,000 movies made by 6,040 MovieLens users who joined MovieLens in 2000. The file was obtained from https://grouplens.org/datasets/movielens/1m/. This data was collected by The GroupLens Research Project, a research group in the Department of Computer Science and Engineering at the University of Minnesota. In addition, the MovieLens dataset includes two additional files: one containing movie titles and another with users’ sociodemographic information. 

First, let's import the dataset ratings.dat

In [ ]:
# load the movie ratings
df_ratings = pd.read_csv('/Users/svazq/HES/CSCI_S89/ratings.dat', sep='::', engine='python', names=['user_id', 'movie_id', 'rating', 'timestamp'])

In [ ]:
df_ratings.head()

In [ ]:
df_ratings.info()

In [ ]:
df_ratings.shape

In [ ]:
# Let's check how many users and movies there are
df_ratings['user_id'].nunique(), df_ratings['movie_id'].nunique(), len(df_ratings)

There are 1,000,209 ratings from 6,040 users on 3,706 movies.

In [ ]:
# Check ratings distribution
rating_dist = df_ratings['rating'].value_counts(normalize=True).sort_index()

# Plot distribution
plt.figure(figsize=(6, 4))
rating_dist.plot(kind='bar')
plt.title('Ratings Distribution')
plt.xlabel('Rating')
plt.ylabel('% of total ratings')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

We can observe the distribution is skewed toward 3–5 stars with more values on '4' stars.

In [ ]:
user_activity = df_ratings.groupby('user_id')['movie_id'].count()
user_activity.describe()

On average, each user rated about 166 movies, but the standard deviation is large, so there is a big variation in how many ratings users have given. Using other descriptive statistics we can observe:
- The least active users have rated 20 movies,
- 25% of users rated 44 movies or fewer.
- Half of the users rated fewer than 96 movies.
- 25% of users rated more than 208 movies.
- The most active user rated 2,314 movies.

In [ ]:
# Let's plot a histogram of ratings per user
plt.hist(user_activity, bins=50)
plt.xlabel("Number of ratings per user")
plt.ylabel("Count of users")
plt.show()

The histogram is highly skewed, with most users rating few movies and a small group of users rating hundreds or thousands.

In [ ]:
# Map user and movie IDs to indices
user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()

df_ratings['user_idx'] = user_encoder.fit_transform(df_ratings['user_id'])
df_ratings['movie_idx'] = movie_encoder.fit_transform(df_ratings['movie_id'])

In [ ]:
# create user-rating matrix
n_users = df_ratings['user_idx'].nunique()
n_movies = df_ratings['movie_idx'].nunique()


In [ ]:
# User-movie matrix
user_item_matrix = np.zeros((n_users, n_movies), dtype=np.float32)

for row in df_ratings.itertuples():
    user_item_matrix[row.user_idx, row.movie_idx] = row.rating

In [ ]:
# Split train/val using a leave one out approach. 
# i.e. for every user, one rating was randomly selected and set aside as the validation example. 
X_train = user_item_matrix.copy()
X_val = np.zeros_like(user_item_matrix)

rng = np.random.default_rng(seed=42)

for user in range(user_item_matrix.shape[0]):
    rated_indices = np.where(user_item_matrix[user] > 0)[0]
    if len(rated_indices) > 1:
        chosen = rng.choice(rated_indices)
        X_val[user, chosen] = user_item_matrix[user, chosen]
        X_train[user, chosen] = 0


In [ ]:
# User defined function for the loss function used for training (RMSE)
def rmse(y_true, y_pred):
    return tf.sqrt(tf.reduce_mean(tf.square(y_true - y_pred)))

In [ ]:
# input size
n_movies = user_item_matrix.shape[1]

Now let's build the autoenconder witn an encoder and a decoder. The first encoder layer has 2048 units with ReLU activation, the second layer compresses the information into a 1024 dimensional latent representation. The decoder mirrors this process in reverse, starting from the 1024 latent features and expanding back to 2048 units, and finally reconstructing the full movie rating profile with an output layer of size equal to the number of movies. The linear activation in the last layer allows the model to predict ratings directly on the same 1 to 5 scale as the input.


In [ ]:
# Encoder
encoder_v1 = models.Sequential([
    layers.Dense(2048, activation='relu', input_shape=[n_movies]),
    layers.Dense(1024, activation='relu')
])

In [ ]:
# Decoder
decoder_v1 = models.Sequential([
    layers.Dense(2048, activation='relu', input_shape=[1024]),  
    layers.Dense(n_movies, activation='linear')
])

RMSE was selected as the loss function because it directly measures how close the predicted ratings are to the actual ratings on the original scale, making it easy to interpret and appropriate for a regression task like rating prediction. 

In [ ]:
autoencoder_v1 = models.Sequential([encoder_v1, decoder_v1])
autoencoder_v1.compile(loss=rmse, optimizer=optimizers.Adam(learning_rate=0.01), metrics=["mae"])
autoencoder_v1.summary()

In [ ]:
# train model
history_v1 = autoencoder_v1.fit(
    X_train, X_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, X_val)
)

Let's save model to keep for future Jupyter sessions

In [ ]:
# save model
autoencoder_v1.save('autoencoder_v1.h5')

In [ ]:
# Save history as a numpy file
np.save('training_history_autoencoder_v1.npy', history_v1.history)

In [ ]:
# Plot training and validation loss (RMSE) and metric (MAE)
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# Plot RMSE
axs[0].plot(history_v1.history['loss'], label='Train RMSE')
axs[0].plot(history_v1.history['val_loss'], label='Val RMSE')
axs[0].set_title('RMSE over Epochs')
axs[0].set_xlabel('Epochs')
axs[0].set_ylabel('RMSE')
axs[0].legend()

# Plot MAE
axs[1].plot(history_v1.history['mae'], label='Train MAE')
axs[1].plot(history_v1.history['val_mae'], label='Val MAE')
axs[1].set_title('MAE over Epochs')
axs[1].set_xlabel('Epochs')
axs[1].set_ylabel('MAE')
axs[1].legend()

plt.tight_layout()
plt.show()

The plots show that the model initially learns well, with both training and validation errors dropping quickly and staying low. However, around epoch 15, the RMSE and MAE begin to rise, indicating overfitting. This suggests we could try early stopping or regularization to help prevent this.

In [ ]:
# let's add Early Stopping

early_stop = EarlyStopping(
    monitor='val_loss',       
    patience=5,              
    restore_best_weights=True 
)

# Encoder
encoder_v2 = models.Sequential([
    layers.Dense(2048, activation='relu', input_shape=[n_movies]),
    layers.Dense(1024, activation='relu')
])
# Decoder
decoder_v2 = models.Sequential([
    layers.Dense(2048, activation='relu', input_shape=[1024]),  
    layers.Dense(n_movies, activation='linear')
])

autoencoder_v2 = models.Sequential([encoder_v2, decoder_v2])
autoencoder_v2.compile(loss=rmse, optimizer=optimizers.Adam(learning_rate=0.01), metrics=["mae"])
autoencoder_v2.summary()

In [ ]:
history_v2 = autoencoder_v2.fit(
    X_train, X_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, X_val),
    callbacks=[early_stop]
)

In [ ]:
# save model
autoencoder_v2.save('autoencoder_v2.h5')

In [ ]:
# Save history as a numpy file
np.save('training_history_autoencoder_v2.npy', history_v2.history)

In [ ]:
# Plot training and validation loss (RMSE) and metric (MAE)
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# Plot RMSE
axs[0].plot(history_v2.history['loss'], label='Train RMSE')
axs[0].plot(history_v2.history['val_loss'], label='Val RMSE')
axs[0].set_title('RMSE over Epochs')
axs[0].set_xlabel('Epochs')
axs[0].set_ylabel('RMSE')
axs[0].legend()

# Plot MAE
axs[1].plot(history_v2.history['mae'], label='Train MAE')
axs[1].plot(history_v2.history['val_mae'], label='Val MAE')
axs[1].set_title('MAE over Epochs')
axs[1].set_xlabel('Epochs')
axs[1].set_ylabel('MAE')
axs[1].legend()

plt.tight_layout()
plt.show()

The model is training smoothly and generalizing well. Both RMSE and MAE drop quickly in the first epoch and then continue to improve gradually. The validation curves stay low and stable throughout, showing no signs of overfitting. In fact, validation error is consistently lower than training error, which suggests the model is performing well on unseen data.

## Additional models

Let's try a model with an additional layer and smaller bottleneck:

In [ ]:
# Encoder
encoder_v3 = models.Sequential([
    layers.Dense(2048, activation='relu', input_shape=[n_movies]),
    layers.Dense(1024, activation='relu'),
    layers.Dense(512, activation='relu')
])

# Decoder
decoder_v3 = models.Sequential([
    layers.Dense(1024, activation='relu', input_shape=[512]),
    layers.Dense(2048, activation='relu'),
    layers.Dense(n_movies, activation='linear') 
])

# Autoencoder 
autoencoder_v3 = models.Sequential([encoder_v3, decoder_v3])

# Compile
autoencoder_v3.compile(loss=rmse, optimizer=optimizers.Adam(learning_rate=0.01), metrics=["mae"])

autoencoder_v3.summary()

In [ ]:
history_v3 = autoencoder_v3.fit(
    X_train, X_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, X_val),
    callbacks=[early_stop]
)

In [ ]:
# save model
autoencoder_v3.save('autoencoder_v3.h5')

In [ ]:
# Save history as a numpy file
np.save('training_history_autoencoder_v3.npy', history_v3.history)

In [ ]:
# Plot training and validation loss (RMSE) and metric (MAE)
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# Plot RMSE
axs[0].plot(history_v3.history['loss'], label='Train RMSE')
axs[0].plot(history_v3.history['val_loss'], label='Val RMSE')
axs[0].set_title('RMSE over Epochs')
axs[0].set_xlabel('Epochs')
axs[0].set_ylabel('RMSE')
axs[0].legend()

# Plot MAE
axs[1].plot(history_v3.history['mae'], label='Train MAE')
axs[1].plot(history_v3.history['val_mae'], label='Val MAE')
axs[1].set_title('MAE over Epochs')
axs[1].set_xlabel('Epochs')
axs[1].set_ylabel('MAE')
axs[1].legend()

plt.tight_layout()
plt.show()

This training looks very stable and effective. Both RMSE and MAE drop quickly after the first epoch and then flatten out smoothly. Validation performance remains consistently low across all epochs, indicating the model is generalizing well without overfitting. However, the metrics from the previous model were smaller.

Now let's try a model with L2 regularization:

In [ ]:
# L2 regularization factor
l2_reg = regularizers.l2(1e-4)

# Encoder with L2 regularization
encoder_v4 = models.Sequential([
    layers.Dense(2048, activation='relu', input_shape=[n_movies],kernel_regularizer=l2_reg),
    layers.Dense(1024, activation='relu', kernel_regularizer=l2_reg),
    layers.Dense(512, activation='relu', kernel_regularizer=l2_reg)
])

# Decoder with L2 regularization
decoder_v4 = models.Sequential([
    layers.Dense(1024, activation='relu', input_shape=[512],
                 kernel_regularizer=l2_reg),
    layers.Dense(2048, activation='relu', kernel_regularizer=l2_reg),
    layers.Dense(n_movies, activation='linear')  
])

# Autoencoder with L2
autoencoder_v4 = models.Sequential([encoder_v4, decoder_v4])

# Compile
autoencoder_v4.compile(loss=rmse, optimizer=optimizers.Adam(learning_rate=0.01), metrics=["mae"])

autoencoder_v4.summary()

In [ ]:
history_v4 = autoencoder_v4.fit(
    X_train, X_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, X_val),
    callbacks=[early_stop]
)

In [ ]:
# save model
autoencoder_v4.save('autoencoder_v4.h5')

In [ ]:
# Save history as a numpy file
np.save('training_history_autoencoder_v4.npy', history_v4.history)

In [ ]:
# Plot training and validation loss (RMSE) and metric (MAE)
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# Plot RMSE
axs[0].plot(history_v4.history['loss'], label='Train RMSE')
axs[0].plot(history_v4.history['val_loss'], label='Val RMSE')
axs[0].set_title('RMSE over Epochs')
axs[0].set_xlabel('Epochs')
axs[0].set_ylabel('RMSE')
axs[0].legend()

# Plot MAE
axs[1].plot(history_v4.history['mae'], label='Train MAE')
axs[1].plot(history_v4.history['val_mae'], label='Val MAE')
axs[1].set_title('MAE over Epochs')
axs[1].set_xlabel('Epochs')
axs[1].set_ylabel('MAE')
axs[1].legend()

plt.tight_layout()
plt.show()

These plots show an improvement. Both RMSE and MAE drop quickly in the first few epochs and continue to decrease steadily throughout the 30 epochs. Validation curves stay low and follow the same smooth downward trend as training, which indicates that the model is learning effectively without overfitting. The gap between training and validation error is small and stable, suggesting good generalization. 

Now let's add dropout layers:

In [ ]:
# L2 regularization factor
l2_reg = regularizers.l2(1e-4)

# Dropout rate
dropout_rate = 0.2

# Encoder 
encoder_v5 = models.Sequential([
    layers.Dense(2048, activation='relu', input_shape=[n_movies], kernel_regularizer=l2_reg),
    layers.Dropout(dropout_rate),
    layers.Dense(1024, activation='relu', kernel_regularizer=l2_reg),
    layers.Dropout(dropout_rate),
    layers.Dense(512, activation='relu', kernel_regularizer=l2_reg),
    layers.Dropout(dropout_rate)
])

# Decoder 
decoder_v5 = models.Sequential([
    layers.Dense(1024, activation='relu', input_shape=[512], kernel_regularizer=l2_reg),
    layers.Dropout(dropout_rate),
    layers.Dense(2048, activation='relu', kernel_regularizer=l2_reg),
    layers.Dropout(dropout_rate),
    layers.Dense(n_movies, activation='linear')  
])

# Autoencoder 
autoencoder_v5 = models.Sequential([encoder_v5, decoder_v5])

# Compile
autoencoder_v5.compile(loss=rmse, optimizer=optimizers.Adam(learning_rate=0.01), metrics=["mae"])
autoencoder_v5.summary()


In [ ]:
history_v5 = autoencoder_v5.fit(
    X_train, X_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, X_val),
    callbacks=[early_stop]
)

In [ ]:
# save model
autoencoder_v5.save('autoencoder_v5.h5')

In [ ]:
# Save history as a numpy file
np.save('training_history_autoencoder_v5.npy', history_v5.history)

In [ ]:
# Plot training and validation loss (RMSE) and metric (MAE)
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# Plot RMSE
axs[0].plot(history_v5.history['loss'], label='Train RMSE')
axs[0].plot(history_v5.history['val_loss'], label='Val RMSE')
axs[0].set_title('RMSE over Epochs')
axs[0].set_xlabel('Epochs')
axs[0].set_ylabel('RMSE')
axs[0].legend()

# Plot MAE
axs[1].plot(history_v5.history['mae'], label='Train MAE')
axs[1].plot(history_v5.history['val_mae'], label='Val MAE')
axs[1].set_title('MAE over Epochs')
axs[1].set_xlabel('Epochs')
axs[1].set_ylabel('MAE')
axs[1].legend()

plt.tight_layout()
plt.show()

These plots show consistent decreases in both RMSE and MAE for training and validation in early epochs, but after 10 epochs, the model validation metrics start oscilating so it does not look as stable as the previous model.

### Autoencoder Model Comparison

| Model        | Description             | Best Val RMSE | Best Val MAE | Epoch of Best Val RMSE |
|--------------|-------------------------|----------------|----------------|--------|
| Model 1  | Baseline | 0.1046         | 0.0415         | 12     |
| Model 2  | Early stopping              | 0.0978         | 0.0334     | 3      |
| Model 3  | Early stopping + Smaller bottleneck            | 0.1089         | 0.0451         | 5      |
| Model 4  | Early stopping + Smaller bottleneck + L2 regularization          | 0.0956     | 0.0357         | 29     |
| Model 5  | Early stopping + Smaller bottleneck + L2 regularization +  dropout layers         | 0.1431         | 0.0401         | 17    |

Now lets compare in a graph the Best validation RMSEs:

In [ ]:
# Model names and their best validation RMSEs
models = ['Model 1', 'Model 2', 'Model 3', 'Model 4', "Model 5"]
accuracies = [0.1046, 0.0978, 0.1089, 0.0956, 0.1431]

# Create bar plot
plt.figure(figsize=(10, 6))
plt.bar(models, accuracies)
plt.title('Best Validation RMSE')
plt.ylabel('RMSE')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

Model 4 achieved the best validation RMSE and second best MAE , showing that combining early stopping with a smaller bottleneck and L2 regularization improves generalization. Model 2 also performed well with early stopping alone, reaching strong results in just 3 epochs. Model 5, which added dropout layers, showed worse performance, likely due to excessive regularization. Model 3 underperformed and Model 1 served as a solid baseline but was outperformed by regularized models. Overall, regularization helped, but too much hurt performance.

For the final model I will select Model 4.

## Inference

Now let's recommend movies to a user by predicting ratings with the trained model. First lets exclude movies the user has already rated and then show the top 10 unseen movies with the highest predicted scores.


In [ ]:
# Let's generate recommendations for a user
user_idx = 77

In [ ]:
# Get the user vector
user_vector = user_item_matrix[user_idx].reshape(1, -1)

In [ ]:
# Predict ratings using the trained model
predicted_ratings = autoencoder_v4.predict(user_vector)[0]

In [ ]:
# Filter movies already rated by the user
already_rated = user_item_matrix[user_idx] > 0

In [ ]:
# Set predicted ratings for already rated movies to -inf
predicted_ratings[already_rated] = -np.inf


In [ ]:
# Top-N recommendations
top_n = 10
recommended_indices = np.argsort(predicted_ratings)[-top_n:][::-1]

In [ ]:
# Map indices back to movie IDs
for idx in recommended_indices:
    movie_id = movie_encoder.inverse_transform([idx])[0]
    print(f"Recommend movie_id {movie_id} with predicted score {predicted_ratings[idx]:.3f}")

In [ ]:
# load the movie names data with names, to include in the recommendation
df_movies = pd.read_csv('/Users/svazq/HES/CSCI_S89/movies.dat', sep="::", engine="python", names=["movie_id", "title", "genres"], encoding="ISO-8859-1")


In [ ]:
# Map movie_id to title
movie_id_to_title = dict(zip(df_movies["movie_id"], df_movies["title"]))
print(f"Top {top_n} recommendations for user {user_idx}:\n")

for idx in recommended_indices:
    # Original MovieLens ID
    movie_id = movie_encoder.inverse_transform([idx])[0]
    # Movie title
    title = movie_id_to_title.get(movie_id, "Unknown Title")
    score = predicted_ratings[idx]
    print(f"{title} (MovieLens ID: {movie_id}) - predicted rating: {score:.2f}")

In [ ]:
# Let's check what movies were higher rated by this user
user_0_ratings = df_ratings[df_ratings['user_id'] == 77]

top_10_rated = user_0_ratings.sort_values(by='rating', ascending=False).head(10)

# Merge with movie titles 
top_10_with_titles = top_10_rated.merge(df_movies, on='movie_id')

# Display top 10 movies
print(top_10_with_titles[['movie_id', 'title', 'rating']])

## Corrected evaluation: masked RMSE
Corrected evaluation for the Movie Recommender autoencoder (CSCI S-89).

THE BUG
-------
The notebook trains with fit(X_train, X_train, validation_data=(X_val, X_val))
using a plain RMSE over all 3,706 entries per user. But X_val contains exactly
ONE real rating per user -- the other ~3,705 entries are zeros for unrated
movies, which the model trivially predicts as ~0. The single real rating's
error is therefore diluted ~3,706x, which is why the reported validation RMSE
(~0.0956) is ~9x better than the best published MovieLens 1M result (~0.85)
and why validation error sits *below* training error. The metric measures
"how well the model outputs zeros", not rating-prediction accuracy.

THE FIX
-------
Score ONLY actually-rated entries (ratings are 1-5, never 0, so nonzero ==
rated). Two drop-in pieces below:

  Part A -- masked loss/metric for (re)training. Makes both the training
            loss and the validation metric honest. Early stopping on
            val_masked_rmse then actually means something.

  Part B -- honest re-evaluation of your ALREADY-TRAINED models with NO
            retraining. Rebuilds the identical leave-one-out split
            (same seed=42, same loop order) and scores each model only on
            the held-out ratings. Run this in your Jupyter env where the
            trained models and data live.

Paste each part into its own notebook cell, in order.


In [ ]:
# Part A -- masked loss/metric for (re)training
-- masked RMSE / MAE for training (drop-in replacements)
# ======================================================================
import tensorflow as tf


def masked_rmse(y_true, y_pred):
    """RMSE computed only over actually-rated entries (nonzero y_true)."""
    mask = tf.cast(tf.not_equal(y_true, 0.0), tf.float32)
    sq_err = tf.square((y_true - y_pred) * mask)
    denom = tf.maximum(tf.reduce_sum(mask), 1.0)
    return tf.sqrt(tf.reduce_sum(sq_err) / denom)


def masked_mae(y_true, y_pred):
    """MAE computed only over actually-rated entries (nonzero y_true)."""
    mask = tf.cast(tf.not_equal(y_true, 0.0), tf.float32)
    abs_err = tf.abs((y_true - y_pred) * mask)
    denom = tf.maximum(tf.reduce_sum(mask), 1.0)
    return tf.reduce_sum(abs_err) / denom


# Use it when compiling (replaces loss=rmse, metrics=["mae"]):
#   autoencoder.compile(loss=masked_rmse,
#                       optimizer=optimizers.Adam(learning_rate=0.01),
#                       metrics=[masked_mae])
# And point EarlyStopping at the honest metric:
#   early_stop = callbacks.EarlyStopping(monitor="val_masked_rmse",
#                                        patience=3, restore_best_weights=True)


# ======================================================================


In [ ]:
# Part B -- honest re-evaluation of trained models (NO retraining)
# ======================================================================
# Run this AFTER the cells that build `user_item_matrix` (it only needs
# the matrix and a trained model object, e.g. autoencoder_v4).
import numpy as np


def rebuild_leave_one_out(user_item_matrix, seed=42):
    """Rebuild the notebook's exact split: fresh RNG, same order -> identical."""
    rng = np.random.default_rng(seed=seed)
    held_out = []  # (user_idx, movie_idx, true_rating)
    X_in = user_item_matrix.copy()
    for user in range(user_item_matrix.shape[0]):
        rated = np.where(user_item_matrix[user] > 0)[0]
        if len(rated) > 1:
            chosen = rng.choice(rated)
            held_out.append((user, int(chosen), float(user_item_matrix[user, chosen])))
            X_in[user, chosen] = 0.0
    return X_in, held_out


def honest_metrics(model, X_in, held_out, batch_size=256):
    """RMSE/MAE scored ONLY on the held-out ratings. No retraining."""
    preds = model.predict(X_in, batch_size=batch_size, verbose=0)
    se, ae = 0.0, 0.0
    for u, m, r in held_out:
        err = r - float(preds[u, m])
        se += err * err
        ae += abs(err)
    n = len(held_out)
    return float(np.sqrt(se / n)), float(ae / n)


# ---- usage: score all five trained models on the honest metric ----
# X_in, held_out = rebuild_leave_one_out(user_item_matrix)
# print(f"Held-out ratings: {len(held_out)}")
# for name, model in [("Model 1", autoencoder_v1),
#                     ("Model 2", autoencoder_v2),
#                     ("Model 3", autoencoder_v3),
#                     ("Model 4", autoencoder_v4),
#                     ("Model 5", autoencoder_v5)]:
#     rmse, mae = honest_metrics(model, X_in, held_out)
#     print(f"{name}: honest val RMSE={rmse:.4f}  MAE={mae:.4f}")
#
# EXPECT: RMSE around 0.85-1.0 (the credible range for this setup on
# MovieLens 1M), NOT ~0.10. The model ranking may change -- that is the
# honest comparison to put on the portfolio.
